In [ ]:
# Load all the necessary libraries
import pandas as pd
import seaborn as sns

# Supress all warnings (bcoz they're quite annoying)
import warnings

warnings.filterwarnings("ignore")

import sys
sys.path.append("..")

In [ ]:
def get_class_palette():
    
    colors = sns.color_palette("colorblind")
    colors2 = sns.color_palette("tab20")
    pal = {
        "young": colors[2],
        "aged": colors[3],
        "aged_CASIN": colors[0],
        "aged_CDK8i": colors[1],
        "aged_DMSO": colors[4],
        "aged_IOX": colors[5],
        "aged_UNC": colors[6],
        "aged_treated_RhoAi": colors[7],
        "aged_RhoAi": colors[7],
        "middle": colors[8],
        "myeloid_progenitors": colors[9],
        "young_treated_NaB": colors2[8],
    }
    
    return pal

In [ ]:
nuc_df = pd.read_csv(
    "../results/pyradiomics/nobin_0.1_z_score_all_df.csv",
    index_col=0,
)

pal = get_class_palette()

In [ ]:
my_df = nuc_df
condition = my_df.condition

czi_metadata_filt = pd.read_csv("../results/czi_metadata_filt.csv")
my_df = my_df.merge(czi_metadata_filt, left_on="czi_path", right_on="path")
my_df["condition"] = condition

In [ ]:
# Filter those features that belong to the filter WAVELET H
my_df = my_df[[col for col in my_df.columns if not col.startswith("wavelet-H")]]

In [ ]:
features = my_df.columns[:-36]
features

In [ ]:
my_df.describe().T

In [ ]:
# Drop columns that contain NaN values
# nuc_df = nuc_df.dropna()
sns.set(rc={"figure.figsize": (20, 16)})
sns.set_style("whitegrid")

int_df = my_df[features]

# Drop all the columns that have the same value for all rows
nunique = int_df.nunique()
cols_drop = nunique[nunique == 1].index
int_df = int_df.drop(cols_drop, axis=1)

In [ ]:
# Z-score normalization
norm_df = int_df.copy()
norm_df = (int_df - int_df.mean()) / int_df.std()

In [ ]:
norm_df.columns

In [ ]:
norm_df["condition"] = condition
norm_df["acquired_by"] = my_df["acquired_by"]
stds_by_cond = norm_df.groupby(['acquired_by', 'condition']).std()
avg_std_per_condition = stds_by_cond.mean(axis=1)

In [ ]:
cond_dirs = [
    "aged",
    "young",
    "aged_DMSO",
    "aged_CASIN",
    "aged_CDK8i",
    "aged_IOX",
    "aged_UNC",
    "aged_treated_RhoAi",
]


my_df = my_df[my_df["condition"].isin(cond_dirs)]

In [ ]:
# Merge aged and aged + DMSO
norm_df["condition"] = norm_df["condition"].replace("aged_DMSO", "aged")
norm_df["condition"] = norm_df["condition"].replace("aged_treated_RhoAi", "aged_RhoAi")
cond_dirs = [
    "aged",
    "aged_CASIN",
    "aged_IOX",
    "aged_UNC",
    "aged_RhoAi",
    "young",
]

# Only include specified conditions
norm_df = norm_df[norm_df["condition"].isin(cond_dirs)]

In [ ]:
row_colors

In [ ]:
shap_feats = [
    'wavelet-LHH_firstorder_Median',
    'original_ngtdm_Complexity',
    'log-sigma-0-5-mm-3D_glcm_SumEntropy',
    'log-sigma-0-5-mm-3D_glszm_SizeZoneNonUniformity',
    'wavelet-LHH_firstorder_Kurtosis',
    'log-sigma-0-5-mm-3D_glszm_SmallAreaHighGrayLevelEmphasis',
    'original_shape_LeastAxisLength',
    'wavelet-LLH_glcm_SumEntropy',
    'wavelet-LHH_glszm_ZoneVariance',
    'log-sigma-0-5-mm-3D_firstorder_Minimum',
    'wavelet-LLL_glszm_ZoneVariance',
    'log-sigma-0-5-mm-3D_firstorder_Range',
    'log-sigma-1-0-mm-3D_firstorder_Maximum',
    'wavelet-LHL_glcm_SumEntropy'
]

In [ ]:
norm_df = norm_df[norm_df["condition"].isin(cond_dirs)]
norm_df = norm_df.sort_values(by=["condition"])
row_colors = norm_df["condition"].map(pal)
b = sns.clustermap(norm_df[shap_feats], cmap='inferno', annot=False, robust=True,
               figsize=(5, 12), row_colors=row_colors, row_cluster=False, 
               col_cluster=True, dendrogram_ratio=0.1)
b.ax_heatmap.set_yticklabels([])     # Hide the tick labels
b.ax_heatmap.tick_params(right=False) # Optionally remove the ticks too